In [ ]:
from project_config import WEIGHTS_ROOT, OUTPUT_ROOT, TRAIN_SPLIT, VALIDATION_SPLIT, RANDOM_SEED, NUM_WORKERS, DEVICES, STRATEGY, load_split_samples
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import numpy as np
import os
import math
from pytorch_lightning.callbacks import TQDMProgressBar, ModelCheckpoint


In [ ]:
pl.seed_everything(RANDOM_SEED, workers=True)
train_dic_x, train_dic_mask, train_dic_y = load_split_samples(TRAIN_SPLIT)
val_dic_x, val_dic_mask, val_dic_y = load_split_samples(VALIDATION_SPLIT)
with np.load(train_dic_x[0]) as sample:
    in_channels = sample['arr_0'].shape[0]
with np.load(train_dic_y[0]) as sample:
    out_channels = sample['arr_0'].shape[0]


In [ ]:
class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-08):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return normed * self.weight

class SwiGLU_FFN(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        hidden_dim = int(d_model * 8 / 3)
        self.w_g = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w_2 = nn.Linear(hidden_dim, d_model, bias=False)

    def forward(self, x):
        return self.w_2(F.silu(self.w_g(x)) * self.w_1(x))

class WMHDA_WeightGenerator(nn.Module):

    def __init__(self, d_model, h):
        super().__init__()
        hidden_dim = d_model // 2
        self.w1 = nn.Linear(d_model, hidden_dim)
        self.w2 = nn.Linear(hidden_dim, h)

    def forward(self, x):
        x_bar = x.mean(dim=1)
        s = self.w2(F.gelu(self.w1(x_bar)))
        alpha = F.softmax(s, dim=-1)
        return alpha

class WMHDA_DIFFTransformerBlock(nn.Module):

    def __init__(self, d_model, h, layer_idx):
        super().__init__()
        self.d_model = d_model
        self.h = h
        self.d = d_model // (2 * h)
        self.ln1 = RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, h * 2 * self.d, bias=False)
        self.k_proj = nn.Linear(d_model, h * 2 * self.d, bias=False)
        self.v_proj = nn.Linear(d_model, h * 2 * self.d, bias=False)
        self.weight_gen = WMHDA_WeightGenerator(d_model, h)
        self.lambda_init = 0.8 - 0.6 * math.exp(-0.3 * (layer_idx - 1))
        self.lambda_q1 = nn.Parameter(torch.randn(self.d))
        self.lambda_k1 = nn.Parameter(torch.randn(self.d))
        self.lambda_q2 = nn.Parameter(torch.randn(self.d))
        self.lambda_k2 = nn.Parameter(torch.randn(self.d))
        self.rmsn = RMSNorm(2 * self.d)
        self.out_proj = nn.Linear(2 * self.d, d_model, bias=False)
        self.ln2 = RMSNorm(d_model)
        self.swiglu = SwiGLU_FFN(d_model)

    def forward(self, x):
        B, N, D = x.shape
        residual = x
        x_norm = self.ln1(x)
        alpha = self.weight_gen(x_norm)
        q = self.q_proj(x_norm).view(B, N, self.h, 2, self.d).permute(0, 2, 3, 1, 4)
        k = self.k_proj(x_norm).view(B, N, self.h, 2, self.d).permute(0, 2, 3, 1, 4)
        v = self.v_proj(x_norm).view(B, N, self.h, 2 * self.d).transpose(1, 2)
        q1, q2 = (q[:, :, 0], q[:, :, 1])
        k1, k2 = (k[:, :, 0], k[:, :, 1])
        lambda_val = torch.exp(torch.dot(self.lambda_q1, self.lambda_k1)) - torch.exp(torch.dot(self.lambda_q2, self.lambda_k2)) + self.lambda_init
        scale = 1.0 / math.sqrt(self.d)
        attn1 = F.softmax(q1 @ k1.transpose(-2, -1) * scale, dim=-1)
        attn2 = F.softmax(q2 @ k2.transpose(-2, -1) * scale, dim=-1)
        diff_attn = attn1 - lambda_val * attn2
        head_out = diff_attn @ v
        head_out_normed = self.rmsn(head_out)
        H_bar = head_out_normed * (1.0 - self.lambda_init)
        alpha = alpha.unsqueeze(-1).unsqueeze(-1)
        H_sum = (alpha * H_bar).sum(dim=1)
        attn_out = self.out_proj(H_sum)
        x = residual + attn_out
        x = x + self.swiglu(self.ln2(x))
        return x

class DoubleConv(nn.Module):

    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True), nn.Conv2d(out_c, out_c, kernel_size=3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)

class DeconvUp(nn.Module):

    def __init__(self, in_c, out_c):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_c, out_c, kernel_size=2, stride=2)

    def forward(self, x):
        return self.up(x)

class HailPre(nn.Module):

    def __init__(self, in_channels=102, num_frames_out=20, d_model=256, h=4, patch_size=16):
        super().__init__()
        self.patch_size = patch_size
        self.d_model = d_model
        self.patch_embed = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        self.transformer_layers = nn.ModuleList([WMHDA_DIFFTransformerBlock(d_model=d_model, h=h, layer_idx=i + 1) for i in range(12)])
        self.deconv_z12 = DeconvUp(d_model, d_model // 2)
        self.conv_z9 = DoubleConv(d_model, d_model // 2)
        self.up_9_to_8 = DeconvUp(d_model // 2, d_model // 4)
        self.deconv_z6_stage1 = DeconvUp(d_model, d_model // 2)
        self.conv_z6 = DoubleConv(d_model // 2, d_model // 4)
        self.up_6_to_4 = DeconvUp(d_model // 4, d_model // 8)
        self.deconv_z3_stage1 = DeconvUp(d_model, d_model // 2)
        self.conv_z3_stage1 = DoubleConv(d_model // 2, d_model // 2)
        self.deconv_z3_stage2 = DeconvUp(d_model // 2, d_model // 4)
        self.conv_z3 = DoubleConv(d_model // 4, d_model // 8)
        self.up_3_to_2 = DeconvUp(d_model // 8, d_model // 16)
        self.conv_in = DoubleConv(in_channels, d_model // 16)
        self.decoder_stage1 = DoubleConv(d_model, d_model // 2)
        self.decoder_stage2 = DoubleConv(d_model // 2, d_model // 4)
        self.decoder_stage3 = DoubleConv(d_model // 4, d_model // 8)
        self.decoder_stage4 = DoubleConv(d_model // 8, d_model // 16)
        self.final_conv = nn.Conv2d(d_model // 16, num_frames_out, kernel_size=1)

    def forward(self, x):
        B, C_in, H_ori, W_ori = x.shape
        P = self.patch_size
        pad_h = (P - H_ori % P) % P
        pad_w = (P - W_ori % P) % P
        if pad_h > 0 or pad_w > 0:
            x_padded = F.pad(x, (0, pad_w, 0, pad_h))
        else:
            x_padded = x
        B, _, H, W = x_padded.shape
        feat_in = self.conv_in(x_padded)
        patches = self.patch_embed(x_padded)
        seq = patches.flatten(2).transpose(1, 2)
        N = seq.shape[1]
        device = seq.device
        pos = torch.arange(N, device=device, dtype=torch.float32).unsqueeze(1)
        dim = torch.arange(self.d_model, device=device, dtype=torch.float32).unsqueeze(0)
        pos_embed = torch.sin(pos / 10000 ** (2 * (dim // 2) / self.d_model))
        seq = seq + pos_embed.unsqueeze(0)
        features = {}
        for i, layer in enumerate(self.transformer_layers):
            seq = layer(seq)
            if i + 1 in [3, 6, 9, 12]:
                features[f'Z{i + 1}'] = seq.transpose(1, 2).view(B, -1, H // P, W // P)
        Z3, Z6, Z9, Z12 = (features['Z3'], features['Z6'], features['Z9'], features['Z12'])
        z12_up = self.deconv_z12(Z12)
        z9_conv = self.conv_z9(Z9)
        z9_up = F.interpolate(z9_conv, scale_factor=2, mode='bilinear', align_corners=False)
        d1 = self.decoder_stage1(torch.cat([z12_up, z9_up], dim=1))
        d1_up = self.up_9_to_8(d1)
        z6_up = self.deconv_z6_stage1(Z6)
        z6_up = F.interpolate(z6_up, scale_factor=2, mode='bilinear', align_corners=False)
        z6_conv = self.conv_z6(z6_up)
        d2 = self.decoder_stage2(torch.cat([d1_up, z6_conv], dim=1))
        d2_up = self.up_6_to_4(d2)
        z3_up = self.deconv_z3_stage2(self.conv_z3_stage1(self.deconv_z3_stage1(Z3)))
        z3_up = F.interpolate(z3_up, scale_factor=2, mode='bilinear', align_corners=False)
        z3_conv = self.conv_z3(z3_up)
        d3 = self.decoder_stage3(torch.cat([d2_up, z3_conv], dim=1))
        d3_up = self.up_3_to_2(d3)
        d4 = self.decoder_stage4(torch.cat([d3_up, feat_in], dim=1))
        out_padded = self.final_conv(d4)
        out = out_padded[:, :, :H_ori, :W_ori]
        return out


In [ ]:
class SequenceDataset(Dataset):

    def __init__(self, file_dic_x_radar, file_dic_x_mask, file_dic_y):
        self.file_dic_x_radar = file_dic_x_radar
        self.file_dic_x_mask = file_dic_x_mask
        self.file_dic_y = file_dic_y

    def __len__(self):
        return len(self.file_dic_x_radar)

    def __getitem__(self, idx):
        input_data = np.load(self.file_dic_x_radar[idx])['arr_0']
        output_data = np.load(self.file_dic_y[idx])['arr_0']
        input_tensor = torch.from_numpy(input_data).float()
        output_tensor = torch.from_numpy(output_data).float()
        return (input_tensor, output_tensor)

class SequenceDataModule(pl.LightningDataModule):

    def __init__(self, batch_size, num_workers):
        super().__init__()
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        self.train_dataset = SequenceDataset(train_dic_x, train_dic_mask, train_dic_y)
        self.val_dataset = SequenceDataset(val_dic_x, val_dic_mask, val_dic_y)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True, persistent_workers=self.num_workers > 0, drop_last=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True, persistent_workers=self.num_workers > 0)

class Loss(nn.Module):

    def __init__(self, alpha, beta):
        super(Loss, self).__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, y_pred, y_true):
        base_temp = torch.square(y_pred - y_true) * ((y_true + self.alpha) / (1 + self.alpha))
        if torch.sum(y_true) > 0:
            temp = base_temp
        else:
            temp = self.beta * base_temp
        divisor = temp.shape[1] * temp.shape[2] * temp.shape[3]
        return torch.sum(temp) / divisor

class LightningModel(pl.LightningModule):

    def __init__(self, alpha, beta):
        super().__init__()
        self.model = HailPre(in_channels=in_channels, num_frames_out=out_channels, d_model=128, h=4)
        self.criterion = Loss(alpha, beta)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.model(inputs)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self.model(inputs)
        val_loss = self.criterion(outputs, labels)
        self.log('val_loss', val_loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001, fused=torch.cuda.is_available())
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.1)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}}

class SilentValidationProgressBar(TQDMProgressBar):

    def init_validation_tqdm(self):
        bar = super().init_validation_tqdm()
        bar.disable = True
        return bar


In [ ]:
dirpath = WEIGHTS_ROOT / 'DIFF_T'
if not os.path.exists(dirpath):
    os.makedirs(dirpath)
checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=3, filename='best-model-{epoch:02d}-{val_loss:.8f}', dirpath=dirpath, save_weights_only=False)
trainer = pl.Trainer(accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=DEVICES, precision='16-mixed' if torch.cuda.is_available() else '32-true', strategy=STRATEGY, max_epochs=100, callbacks=[SilentValidationProgressBar(), checkpoint_callback], benchmark=True)
model = LightningModel(0.1, 10)
datamodule = SequenceDataModule(batch_size=8, num_workers=NUM_WORKERS)
trainer.fit(model, datamodule=datamodule)


In [ ]:
trainer.save_checkpoint(str(dirpath / 'last.ckpt'))
